In [ ]:
import warnings
import yfinance as yf
import pandas as pd
import numpy as np

# Suppress FutureWarnings that originate inside yfinance internals.
# These come from yfinance's own history.py (empty Series dtype) and
# are not actionable from user code — safe to filter permanently.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="yfinance",
)

# Print libary versions
print(f"yfinance version: {yf.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

yfinance version: 1.3.0


In [2]:
ticker_list = ['2267.T', '3435.T', '1846.HK', '6889.HK', '7399.T', '1913.HK', 'VTU.L']


def _to_naive(ts):
    """Convert any Timestamp to tz-naive. Aware => tz_convert(None), naive => passthrough."""
    if hasattr(ts, 'tzinfo') and ts.tzinfo is not None:
        return ts.tz_convert(None)
    return ts


def _scalar(val):
    """Safely extract a scalar from a value that might be a Series or ndarray.

    yfinance occasionally returns a single-element Series instead of a scalar
    for balance sheet line items (e.g. Ordinary Shares Number). Evaluating
    `if series_val` raises 'truth value of a Series is ambiguous' and crashes
    the whole ticker. This helper collapses it to a plain Python float or None.
    """
    if val is None:
        return None
    if isinstance(val, pd.Series):
        val = val.iloc[0] if not val.empty else None
    if isinstance(val, np.ndarray):
        val = val.flat[0] if val.size > 0 else None
    if val is not None and pd.isna(val):
        return None
    return val


def pull_yf_ticker_data(ticker_list: list):
    """Pull all available annual balance sheet + income data from Yahoo Finance.
    Returns a multi-row DataFrame structured by Ticker and Year.
    """
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching historical data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # --- Currencies ---
            raw_trading = info.get("currency", None)
            raw_financial = info.get("financialCurrency", None)
            trading_curr = raw_trading.upper() if raw_trading else None
            financial_curr = raw_financial.upper() if raw_financial else None

            market_cap = info.get("marketCap", None)

            # --- FX rate ---
            fx_rate = 1.0
            if trading_curr and financial_curr and trading_curr != financial_curr:
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                try:
                    fx_data = yf.Ticker(fx_ticker_str).history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                except Exception:
                    pass

            market_cap_converted = (market_cap * fx_rate) if market_cap is not None else None

            # --- Historical shares outstanding series ---
            shares_series = None
            try:
                shares_series = ticker.get_shares_full(start="2015-01-01", end=None)
            except Exception:
                pass

            # --- TTM Net Income ---
            ttm_net_inc = None
            try:
                ttm_inc = ticker.ttm_income_stmt
                if isinstance(ttm_inc, tuple): ttm_inc = ttm_inc[0]
                if not isinstance(ttm_inc, pd.DataFrame): ttm_inc = pd.DataFrame(ttm_inc)
                if not ttm_inc.empty and "Net Income" in ttm_inc.index:
                    ttm_net_inc = _scalar(ttm_inc.loc["Net Income"].iloc[0])
            except Exception:
                pass

            if ttm_net_inc is None:
                try:
                    q_inc = ticker.quarterly_income_stmt
                    if isinstance(q_inc, tuple): q_inc = q_inc[0]
                    if not isinstance(q_inc, pd.DataFrame): q_inc = pd.DataFrame(q_inc)
                    if not q_inc.empty and "Net Income" in q_inc.index:
                        ttm_net_inc = q_inc.loc["Net Income"].iloc[:4].sum()
                        if pd.isna(ttm_net_inc): ttm_net_inc = None
                except Exception:
                    pass

            # --- Annual statements ---
            raw_bs = ticker.balance_sheet
            raw_inc = ticker.income_stmt
            if isinstance(raw_bs, tuple): raw_bs = raw_bs[0]
            if isinstance(raw_inc, tuple): raw_inc = raw_inc[0]
            annual_bs = raw_bs if isinstance(raw_bs, pd.DataFrame) else pd.DataFrame(raw_bs)
            annual_inc = raw_inc if isinstance(raw_inc, pd.DataFrame) else pd.DataFrame(raw_inc)

            if not annual_bs.empty:
                for report_date in annual_bs.columns:
                    date_str = str(report_date.date())
                    year_val = report_date.year
                    bs_col = annual_bs[report_date]
                    report_date_naive = _to_naive(report_date)

                    total_assets              = _scalar(bs_col.get("Total Assets"))
                    current_assets            = _scalar(bs_col.get("Current Assets"))
                    total_liabilities         = _scalar(bs_col.get("Total Liabilities Net Minority Interest"))
                    total_goodwill_intangibles= _scalar(bs_col.get("Goodwill And Other Intangible Assets"))
                    total_equity              = _scalar(bs_col.get("Common Stock Equity"))
                    total_investments         = _scalar(
                        bs_col.get("Investment Properties") or bs_col.get("Investments And Advances")
                    )

                    # --- Historical Market Cap ---
                    hist_market_cap = None

                    # Strategy A: get_shares_full time series
                    shares_outstanding = None
                    if shares_series is not None and not shares_series.empty:
                        try:
                            closest_share_date = min(
                                shares_series.index,
                                key=lambda x: abs(_to_naive(x) - report_date_naive),
                            )
                            if abs((_to_naive(closest_share_date) - report_date_naive).days) <= 180:
                                shares_outstanding = _scalar(shares_series.loc[closest_share_date])
                        except Exception:
                            pass

                    # Strategy B: Ordinary Shares Number from balance sheet
                    if shares_outstanding is None:
                        shares_outstanding = _scalar(bs_col.get("Ordinary Shares Number"))

                    # Compute hist market cap from shares + historical close price
                    # _scalar() ensures shares_outstanding is a plain Python scalar,
                    # so the `is not None` check is safe — no Series ambiguity.
                    if shares_outstanding is not None:
                        try:
                            start_search = report_date_naive - pd.Timedelta(days=3)
                            end_search   = report_date_naive + pd.Timedelta(days=4)
                            price_hist = ticker.history(start=start_search, end=end_search)
                            if not price_hist.empty:
                                price_hist.index = price_hist.index.map(_to_naive)
                                closest_price_idx = min(
                                    price_hist.index,
                                    key=lambda x: abs(x - report_date_naive),
                                )
                                close_price = price_hist.loc[closest_price_idx, "Close"]
                                hist_market_cap = shares_outstanding * close_price * fx_rate
                        except Exception:
                            pass

                    # --- Income statement matching ---
                    fy_net_inc = None
                    if not annual_inc.empty:
                        if report_date in annual_inc.columns:
                            fy_net_inc = _scalar(annual_inc[report_date].get("Net Income"))
                        else:
                            closest_col = min(
                                annual_inc.columns,
                                key=lambda x: abs(x - report_date),
                            )
                            if abs((closest_col - report_date).days) <= 7:
                                fy_net_inc = _scalar(annual_inc[closest_col].get("Net Income"))

                    is_latest = report_date == annual_bs.columns[0]
                    final_market_cap = market_cap_converted if is_latest else hist_market_cap
                    if is_latest and final_market_cap is None:
                        final_market_cap = hist_market_cap

                    extracted_data.append({
                        "Ticker":                      ticker_str,
                        "Year":                        year_val,
                        "Report Date":                 date_str,
                        "Trading Currency":            trading_curr,
                        "Financial Currency":          financial_curr,
                        "Market Cap":                  final_market_cap,
                        "Total Assets":                total_assets,
                        "Total Current Assets":        current_assets,
                        "Total Goodwill and Intangibles": total_goodwill_intangibles,
                        "Total Liabilities":           total_liabilities,
                        "Total Equity":                total_equity,
                        "Total Investments":           total_investments,
                        "Latest FY Net Income":        fy_net_inc,
                        "TTM Net Income":              ttm_net_inc if is_latest else None,
                    })
            else:
                print(f"   No annual balance sheet found for {ticker_str}")

        except Exception as e:
            print(f"Error fetching data for {ticker_str}: {e}")

    df = pd.DataFrame(extracted_data)
    if not df.empty:
        df = df.sort_values(by=["Ticker", "Year"], ascending=[True, False]).reset_index(drop=True)
    return df

In [3]:
def format_currency(val):
    """Format large numbers into readable strings. Returns 'N/A' for missing data."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return "N/A"
    abs_val = abs(val)
    if abs_val >= 1_000_000_000_000:
        return f"{val / 1_000_000_000_000:.1f}t"
    elif abs_val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.1f}b"
    elif abs_val >= 1_000_000:
        return f"{val / 1_000_000:.1f}m"
    elif abs_val >= 1_000:
        return f"{val / 1_000:.1f}k"
    return str(int(val)) if val == int(val) else f"{val:.1f}"


def format_ratio(val):
    if pd.isna(val) or val < 0 or val in (float("inf"), float("-inf")):
        return "N/A"
    return f"{val:.2f}x"


def format_percentage(val):
    if pd.isna(val) or val in (float("inf"), float("-inf")):
        return "N/A"
    return f"{val:.2%}"

In [4]:
df = pull_yf_ticker_data(ticker_list=ticker_list)


def create_calcs(df_):
    df_ = df_.sort_values(by=["Ticker", "Year"], ascending=[True, True]).copy()

    # Only fill columns where 0 is a correct default (nothing held = 0).
    # All others stay NaN so ratios propagate NaN => 'N/A' after formatting.
    df_["Total Investments"]              = df_["Total Investments"].fillna(0)
    df_["Total Goodwill and Intangibles"] = df_["Total Goodwill and Intangibles"].fillna(0)

    df_ = df_.assign(
        NCAV=lambda d: d["Total Current Assets"] - d["Total Liabilities"],
        NCAV_Inv=lambda d: d["Total Current Assets"] - d["Total Liabilities"] + d["Total Investments"],
        Book_Value=lambda d: d["Total Assets"] - d["Total Liabilities"],
        Tangible_Book_Value=lambda d: (
            d["Total Assets"] - d["Total Liabilities"] - d["Total Goodwill and Intangibles"]
        ),
        ROE=lambda d: d["Latest FY Net Income"] / d["Total Equity"].replace(0, np.nan),
        Price_Book_Ratio=lambda d:          d["Market Cap"] / d["Book_Value"].replace(0, np.nan),
        Price_Tangible_Book_Ratio=lambda d: d["Market Cap"] / d["Tangible_Book_Value"].replace(0, np.nan),
        Price_NCAV_Ratio=lambda d:          d["Market Cap"] / d["NCAV"].replace(0, np.nan),
        Price_NCAV_Inv_Ratio=lambda d:      d["Market Cap"] / d["NCAV_Inv"].replace(0, np.nan),
        PE_Ratio=lambda d:                  d["Market Cap"] / d["Latest FY Net Income"].replace(0, np.nan),
        PE_Ratio_TTM=lambda d:              d["Market Cap"] / d["TTM Net Income"].replace(0, np.nan),
    )

    df_ = df_.sort_values(by=["Ticker", "Year"], ascending=[True, False]).reset_index(drop=True)

    currency_cols = [
        "Market Cap", "Total Assets", "Total Current Assets", "Total Liabilities",
        "Total Investments", "Total Goodwill and Intangibles", "Total Equity",
        "NCAV", "NCAV_Inv", "Book_Value", "Tangible_Book_Value",
        "Latest FY Net Income", "TTM Net Income",
    ]
    for col in currency_cols:
        if col in df_.columns:
            df_[col] = df_[col].map(format_currency)

    ratio_cols = [
        "Price_Book_Ratio", "Price_Tangible_Book_Ratio",
        "Price_NCAV_Ratio", "Price_NCAV_Inv_Ratio",
        "PE_Ratio", "PE_Ratio_TTM",
    ]
    for col in ratio_cols:
        if col in df_.columns:
            df_[col] = df_[col].map(format_ratio)

    for col in ["ROE", "ROE_5YR"]:
        if col in df_.columns:
            df_[col] = df_[col].map(format_percentage)

    return df_


df_calculated = create_calcs(df)
df_calculated

Fetching historical data for: 2267.T...
Fetching historical data for: 3435.T...
Fetching historical data for: 1846.HK...
Fetching historical data for: 6889.HK...
Fetching historical data for: 7399.T...
Fetching historical data for: 1913.HK...
Fetching historical data for: VTU.L...


,Ticker,Year,Report Date,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,...,NCAV_Inv,Book_Value,Tangible_Book_Value,ROE,Price_Book_Ratio,Price_Tangible_Book_Ratio,Price_NCAV_Ratio,Price_NCAV_Inv_Ratio,PE_Ratio,PE_Ratio_TTM
0,1846.HK,2025,2025-12-31,HKD,HKD,869.0m,2.0b,776.7m,400.0m,667.8m,...,108.9m,1.3b,884.5m,4.36%,0.68x,0.98x,7.98x,7.98x,15.96x,15.96x
1,1846.HK,2024,2024-12-31,HKD,HKD,1.3b,1.6b,713.6m,283.7m,473.7m,...,239.9m,1.1b,842.0m,7.53%,1.12x,1.50x,5.26x,5.26x,15.34x,N/A
2,1846.HK,2023,2023-12-31,HKD,HKD,1.7b,1.8b,787.9m,308.7m,588.1m,...,199.8m,1.2b,856.9m,11.57%,1.45x,1.98x,8.49x,8.49x,12.92x,N/A
3,1846.HK,2022,2022-12-31,HKD,HKD,1.7b,1.5b,838.3m,219.7m,497.9m,...,340.4m,1.0b,823.4m,8.82%,1.65x,2.09x,5.06x,5.06x,19.25x,N/A
4,1846.HK,2021,2021-12-31,HKD,HKD,2.5b,N/A,N/A,0,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,1913.HK,2025,2025-12-31,HKD,EUR,9.9b,11.0b,3.0b,1.9b,6.3b,...,-3.3b,4.7b,2.8b,18.34%,2.13x,3.60x,N/A,N/A,11.67x,11.67x
6,1913.HK,2024,2024-12-31,HKD,EUR,16.5b,8.5b,2.5b,867.9m,4.1b,...,-1.6b,4.4b,3.5b,19.36%,3.79x,4.73x,N/A,N/A,19.66x,N/A
7,1913.HK,2023,2023-12-31,HKD,EUR,11.5b,7.6b,2.2b,846.0m,3.7b,...,-1.6b,3.9b,3.0b,17.41%,2.96x,3.78x,N/A,N/A,17.09x,N/A
8,1913.HK,2022,2022-12-31,HKD,EUR,11.2b,7.4b,2.4b,817.8m,3.9b,...,-1.5b,3.5b,2.7b,13.36%,3.19x,4.16x,N/A,N/A,24.00x,N/A
9,1913.HK,2021,2021-12-31,HKD,EUR,12.5b,N/A,N/A,0,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
